In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

# Data loading

In [ ]:
# Set the parent folder that contains raw_data/s1_DR1 and raw_data/s2_A8.
# Prefer the LBROMICS_SCRNA_DATA_ROOT environment variable when set.
import os
main_folder = os.environ.get('LBROMICS_SCRNA_DATA_ROOT', '/pathtoyourfolder/')
if not main_folder.endswith('/'):
    main_folder = main_folder + '/'
print('Using main_folder =', main_folder)

In [ ]:
mtxs = [main_folder+'raw_data/s1_DR1', main_folder+'raw_data/s2_A8']
adatas = [sc.read_10x_mtx(mtx,cache=True) for mtx in mtxs]
adata = adatas[0].concatenate(adatas[1],batch_categories=['WT','mutant'])
adata

In [ ]:
adata.var_names_make_unique()  # this is unnecessary if using `var_names='gene_ids'` in `sc.read_10x_mtx`

# Cell quality control

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [ ]:
for i in range(2):
    adatas[i].var['mt'] = adatas[i].var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
    sc.pp.calculate_qc_metrics(adatas[i], qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    sc.pl.violin(adatas[i], ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
              jitter=0.4, multi_panel=True)

In [ ]:
fig,ax = plt.subplots()
adata[adata.obs.batch=='WT',:].obs.total_counts.hist(bins=200,ax=ax)
plt.show(),plt.close()
fig,ax = plt.subplots()
ax.set_xlim(0,20000)
adata[adata.obs.batch=='WT',:].obs.total_counts.hist(bins=200,ax=ax)
plt.show(),plt.close()

In [ ]:
fig,ax = plt.subplots()
adata[adata.obs.batch=='mutant',:].obs.total_counts.hist(bins=200,ax=ax)
plt.show(),plt.close()
fig,ax = plt.subplots()
ax.set_xlim(0,20000)
adata[adata.obs.batch=='mutant',:].obs.total_counts.hist(bins=200,ax=ax)
plt.show(),plt.close()

In [ ]:
adata.obs.batch.value_counts()

In [ ]:
adatas[0]=adatas[0][adatas[0].obs.total_counts>5000]
adatas[1]=adatas[1][adatas[1].obs.total_counts>5000]

In [ ]:
adatas

In [ ]:
sc.pl.violin(adatas[0], ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
              jitter=0.4, multi_panel=True)
sc.pl.violin(adatas[1], ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
              jitter=0.4, multi_panel=True)

In [ ]:
adata=adatas[0].concatenate(adatas[1:],batch_categories=['WT','mutant'])

In [ ]:
sc.pp.filter_cells(adata, min_genes=1500)
sc.pp.filter_genes(adata, min_cells=10)

In [ ]:
adata = adata[adata.obs.n_genes_by_counts < 8000, :] # too many total counts:
adata

In [ ]:
adata = adata[adata.obs.pct_counts_mt < 10, :] # too many mitochondrial genes 
adata

In [ ]:
df = pd.DataFrame({
    "cell_id": adata.obs_names,
    "QC": 1
})

# save to csv
df.to_csv("good_cells.csv", index=False)